# CineMatch — Fast TMDB Incremental Ingestion

This notebook keeps the existing Bronze/checkpoint architecture but replaces sequential TMDB API calls with controlled concurrent fetching.

**Start with the 100-ID benchmark before running the full Movie ingestion.**


In [ ]:
import importlib
import common

importlib.reload(common)
from common import *

import json
import gzip
import time
import threading
import requests

from concurrent.futures import ThreadPoolExecutor, as_completed
from requests.adapters import HTTPAdapter


## 1. Configuration

Start conservatively. Increase the request rate only after the benchmark is stable.


In [ ]:
BASE_URL = "https://api.themoviedb.org/3"

MOVIE_EXPORT_URL = "https://files.tmdb.org/p/exports/movie_ids_{date}.json.gz"
TV_EXPORT_URL = "https://files.tmdb.org/p/exports/tv_series_ids_{date}.json.gz"

EXPORT_DATE = "11_19_2026"
EXPORT_DATE_DB = "2026-11-19"

BATCH_SIZE = 1000
MAX_WORKERS = 20
REQUESTS_PER_SECOND = 25

MAX_RETRIES = 5
REQUEST_TIMEOUT = (10, 30)

print(f"Batch size: {BATCH_SIZE:,}")
print(f"Workers: {MAX_WORKERS}")
print(f"Target request rate: {REQUESTS_PER_SECOND}/sec")


In [ ]:
engine = get_engine()
print("Database connection ready.")


## 2. Load TMDB export IDs


In [ ]:
def load_export_ids(url):
    ids = set()
    total = 0
    missing = 0
    duplicates = 0

    with requests.get(url, stream=True, timeout=REQUEST_TIMEOUT) as response:
        response.raise_for_status()
        response.raw.decode_content = False

        with gzip.GzipFile(fileobj=response.raw) as file:
            for line in file:
                record = json.loads(line)
                tmdb_id = record.get("id")
                total += 1

                if tmdb_id is None:
                    missing += 1
                elif tmdb_id in ids:
                    duplicates += 1
                else:
                    ids.add(tmdb_id)

    return ids, total, missing, duplicates


In [ ]:
movie_export_url = MOVIE_EXPORT_URL.format(date=EXPORT_DATE)
tv_export_url = TV_EXPORT_URL.format(date=EXPORT_DATE)

movie_ids, movie_total, movie_missing, movie_duplicates = load_export_ids(movie_export_url)

print("MOVIES")
print(f"Unique IDs: {len(movie_ids):,}")
print(f"Total rows: {movie_total:,}")
print(f"Missing IDs: {movie_missing:,}")
print(f"Duplicate IDs: {movie_duplicates:,}")


In [ ]:
tv_ids, tv_total, tv_missing, tv_duplicates = load_export_ids(tv_export_url)

print("TV SHOWS")
print(f"Unique IDs: {len(tv_ids):,}")
print(f"Total rows: {tv_total:,}")
print(f"Missing IDs: {tv_missing:,}")
print(f"Duplicate IDs: {tv_duplicates:,}")


## 3. Bronze and checkpoint state


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT media_type, COUNT(*) AS total_records,
               MIN(tmdb_id) AS min_id, MAX(tmdb_id) AS max_id
        FROM bronze.tmdb_metadata
        GROUP BY media_type
        ORDER BY media_type;
    """))

    for row in result:
        print(row)


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT source, media_type, export_date,
               last_processed_id, records_processed, last_processed_at
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb' AND export_date = :export_date
        ORDER BY media_type;
    """), {"export_date": EXPORT_DATE_DB})

    rows = result.fetchall()
    if rows:
        for row in rows:
            print(row)
    else:
        print("No checkpoint rows found.")


In [ ]:
with engine.connect() as conn:
    existing_movie_ids = set(conn.execute(text("""
        SELECT tmdb_id
        FROM bronze.tmdb_metadata
        WHERE media_type = 'movie'
    """)).scalars().all())

    existing_tv_ids = set(conn.execute(text("""
        SELECT tmdb_id
        FROM bronze.tmdb_metadata
        WHERE media_type = 'tv'
    """)).scalars().all())

print(f"Existing movies: {len(existing_movie_ids):,}")
print(f"Existing TV shows: {len(existing_tv_ids):,}")


## 4. HTTP connection reuse

Each worker keeps its own `requests.Session`, allowing HTTP connections to be reused.


In [ ]:
thread_local = threading.local()

def get_session():
    if not hasattr(thread_local, "session"):
        session = requests.Session()

        adapter = HTTPAdapter(
            pool_connections=MAX_WORKERS,
            pool_maxsize=MAX_WORKERS
        )

        session.mount("https://", adapter)
        session.headers.update({
            "Authorization": f"Bearer {TMDB_API_TOKEN}"
        })

        thread_local.session = session

    return thread_local.session


## 5. Global request-rate limiter


In [ ]:
rate_lock = threading.Lock()
next_request_time = time.monotonic()

def throttle():
    global next_request_time

    with rate_lock:
        now = time.monotonic()
        wait_time = next_request_time - now
        next_request_time = max(now, next_request_time) + 1 / REQUESTS_PER_SECOND

    if wait_time > 0:
        time.sleep(wait_time)


## 6. TMDB detail request with retries


In [ ]:
def get_details(tmdb_id, media_type):
    url = f"{BASE_URL}/{media_type}/{tmdb_id}"

    for attempt in range(MAX_RETRIES):
        throttle()

        try:
            response = get_session().get(url, timeout=REQUEST_TIMEOUT)

            if response.status_code == 200:
                return {"tmdb_id": tmdb_id, "data": response.json(), "error": None}

            if response.status_code == 404:
                return {"tmdb_id": tmdb_id, "data": None, "error": None}

            if response.status_code == 429 or response.status_code >= 500:
                retry_after = response.headers.get("Retry-After")
                wait_time = float(retry_after) if retry_after else min(2 ** attempt, 30)
                time.sleep(wait_time)
                continue

            response.raise_for_status()

        except requests.RequestException as error:
            if attempt == MAX_RETRIES - 1:
                return {"tmdb_id": tmdb_id, "data": None, "error": str(error)}

            time.sleep(min(2 ** attempt, 30))

    return {"tmdb_id": tmdb_id, "data": None, "error": "Maximum retries exceeded"}


In [ ]:
def fetch_batch(batch_ids, media_type, executor):
    future_to_id = {
        executor.submit(get_details, tmdb_id, media_type): tmdb_id
        for tmdb_id in batch_ids
    }

    results = []

    for future in as_completed(future_to_id):
        results.append(future.result())

    return results


## 7. 100-ID benchmark

Run this first. Do not start the full ingestion until this test is stable.


In [ ]:
benchmark_ids = sorted(movie_ids)[:100]

start = time.perf_counter()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    benchmark_results = fetch_batch(benchmark_ids, "movie", executor)

elapsed = time.perf_counter() - start

success = sum(r["data"] is not None for r in benchmark_results)
failed = sum(r["error"] is not None for r in benchmark_results)

print(f"IDs tested: {len(benchmark_ids):,}")
print(f"Successful: {success:,}")
print(f"Failed: {failed:,}")
print(f"Elapsed: {elapsed:.2f} sec")
print(f"Throughput: {len(benchmark_ids) / elapsed:.2f} IDs/sec")

if failed > 2:
    print("WARNING: reduce workers/rate before full ingestion.")
else:
    print("Benchmark acceptable.")


## 8. Calculate remaining Movie IDs

Bronze is the duplicate-prevention source of truth. The checkpoint is used to resume forward.


In [ ]:
with engine.connect() as conn:
    last_processed_id = conn.execute(text("""
        SELECT last_processed_id
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND media_type = 'movie'
          AND export_date = :export_date
    """), {"export_date": EXPORT_DATE_DB}).scalar()

new_movie_ids = sorted(
    tmdb_id for tmdb_id in movie_ids
    if tmdb_id not in existing_movie_ids
    and (last_processed_id is None or tmdb_id > last_processed_id)
)

print(f"Last processed ID: {last_processed_id}")
print(f"Remaining movie IDs: {len(new_movie_ids):,}")


## 9. Database write

Successful records are inserted idempotently. The checkpoint is advanced only for a completely successful batch.


In [ ]:
def save_batch(records, media_type, last_id):
    with engine.begin() as conn:
        if records:
            conn.execute(
                text("""
                    INSERT INTO bronze.tmdb_metadata
                        (media_type, tmdb_id, payload)
                    VALUES
                        (:media_type, :tmdb_id, CAST(:payload AS JSONB))
                    ON CONFLICT (media_type, tmdb_id) DO NOTHING
                """),
                records
            )

        conn.execute(
            text("""
                UPDATE ops.ingestion_checkpoints
                SET last_processed_id = :last_id,
                    records_processed = records_processed + :processed,
                    last_processed_at = NOW(),
                    updated_at = NOW()
                WHERE source = 'tmdb'
                  AND media_type = :media_type
                  AND export_date = :export_date
            """),
            {
                "last_id": last_id,
                "processed": len(records),
                "media_type": media_type,
                "export_date": EXPORT_DATE_DB
            }
        )


## 10. Fast Movie ingestion

A single persistent executor is used for the whole run. Successful records from a failed batch are saved, but the checkpoint is not moved past that batch.


In [ ]:
failed_movie_ids = []
total_start = time.perf_counter()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for i in range(0, len(new_movie_ids), BATCH_SIZE):
        batch_ids = new_movie_ids[i:i + BATCH_SIZE]
        batch_start = time.perf_counter()

        results = fetch_batch(batch_ids, "movie", executor)

        records = []
        batch_failed = []

        for result in results:
            if result["data"] is not None:
                records.append({
                    "media_type": "movie",
                    "tmdb_id": result["tmdb_id"],
                    "payload": json.dumps(result["data"])
                })
            elif result["error"] is not None:
                batch_failed.append(result)

        if records:
            with engine.begin() as conn:
                conn.execute(
                    text("""
                        INSERT INTO bronze.tmdb_metadata
                            (media_type, tmdb_id, payload)
                        VALUES
                            (:media_type, :tmdb_id, CAST(:payload AS JSONB))
                        ON CONFLICT (media_type, tmdb_id) DO NOTHING
                    """),
                    records
                )

        if batch_failed:
            failed_movie_ids.extend(batch_failed)
            print(f"Batch {i // BATCH_SIZE + 1} has {len(batch_failed)} failed IDs.")
            print("Checkpoint NOT advanced. Stop and retry failed IDs.")
            break

        save_batch([], "movie", batch_ids[-1])

        batch_elapsed = time.perf_counter() - batch_start
        total_elapsed = time.perf_counter() - total_start

        print(f"Movie batch {i // BATCH_SIZE + 1} completed")
        print(f"IDs processed: {len(batch_ids):,}")
        print(f"Records fetched: {len(records):,}")
        print(f"Rate: {len(batch_ids) / batch_elapsed:.2f} IDs/sec")
        print(f"Batch time: {batch_elapsed:.1f} sec")
        print(f"Total elapsed: {total_elapsed / 60:.1f} min")
        print(f"Last processed ID: {batch_ids[-1]:,}")
        print("-" * 40)


## 11. Retry failed Movie IDs


In [ ]:
def retry_failed_ids(failed_items, media_type):
    retry_ids = [item["tmdb_id"] for item in failed_items]

    if not retry_ids:
        return []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        return fetch_batch(retry_ids, media_type, executor)


In [ ]:
if failed_movie_ids:
    retry_results = retry_failed_ids(failed_movie_ids, "movie")

    retry_records = [
        {
            "media_type": "movie",
            "tmdb_id": r["tmdb_id"],
            "payload": json.dumps(r["data"])
        }
        for r in retry_results
        if r["data"] is not None
    ]

    still_failed = [
        r for r in retry_results
        if r["error"] is not None
    ]

    if retry_records:
        with engine.begin() as conn:
            conn.execute(
                text("""
                    INSERT INTO bronze.tmdb_metadata
                        (media_type, tmdb_id, payload)
                    VALUES
                        (:media_type, :tmdb_id, CAST(:payload AS JSONB))
                    ON CONFLICT (media_type, tmdb_id) DO NOTHING
                """),
                retry_records
            )

    print(f"Retry successful: {len(retry_records):,}")
    print(f"Still failed: {len(still_failed):,}")
else:
    print("No failed movie IDs.")


## 12. Movie validation


In [ ]:
with engine.connect() as conn:
    row = conn.execute(text("""
        SELECT COUNT(*) AS total_records,
               MIN(tmdb_id) AS min_id,
               MAX(tmdb_id) AS max_id
        FROM bronze.tmdb_metadata
        WHERE media_type = 'movie';
    """)).one()

    print(f"Movie records: {row.total_records:,}")
    print(f"Min ID: {row.min_id:,}")
    print(f"Max ID: {row.max_id:,}")


In [ ]:
with engine.connect() as conn:
    row = conn.execute(text("""
        SELECT last_processed_id, records_processed, last_processed_at
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND media_type = 'movie'
          AND export_date = :export_date;
    """), {"export_date": EXPORT_DATE_DB}).one_or_none()

    print("Movie checkpoint:")
    print(row)


## 13. TV ingestion

Run this after Movie ingestion has been validated. The same optimized fetch layer is reused.


In [ ]:
with engine.connect() as conn:
    last_processed_tv_id = conn.execute(text("""
        SELECT last_processed_id
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND media_type = 'tv'
          AND export_date = :export_date
    """), {"export_date": EXPORT_DATE_DB}).scalar()

new_tv_ids = sorted(
    tmdb_id for tmdb_id in tv_ids
    if tmdb_id not in existing_tv_ids
    and (last_processed_tv_id is None or tmdb_id > last_processed_tv_id)
)

print(f"Last processed TV ID: {last_processed_tv_id}")
print(f"Remaining TV IDs: {len(new_tv_ids):,}")


In [ ]:
failed_tv_ids = []
total_start = time.perf_counter()

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for i in range(0, len(new_tv_ids), BATCH_SIZE):
        batch_ids = new_tv_ids[i:i + BATCH_SIZE]
        batch_start = time.perf_counter()

        results = fetch_batch(batch_ids, "tv", executor)

        records = []
        batch_failed = []

        for result in results:
            if result["data"] is not None:
                records.append({
                    "media_type": "tv",
                    "tmdb_id": result["tmdb_id"],
                    "payload": json.dumps(result["data"])
                })
            elif result["error"] is not None:
                batch_failed.append(result)

        if records:
            with engine.begin() as conn:
                conn.execute(
                    text("""
                        INSERT INTO bronze.tmdb_metadata
                            (media_type, tmdb_id, payload)
                        VALUES
                            (:media_type, :tmdb_id, CAST(:payload AS JSONB))
                        ON CONFLICT (media_type, tmdb_id) DO NOTHING
                    """),
                    records
                )

        if batch_failed:
            failed_tv_ids.extend(batch_failed)
            print(f"TV batch {i // BATCH_SIZE + 1} has {len(batch_failed)} failed IDs.")
            print("Checkpoint NOT advanced.")
            break

        save_batch([], "tv", batch_ids[-1])

        batch_elapsed = time.perf_counter() - batch_start
        total_elapsed = time.perf_counter() - total_start

        print(f"TV batch {i // BATCH_SIZE + 1} completed")
        print(f"IDs processed: {len(batch_ids):,}")
        print(f"Records fetched: {len(records):,}")
        print(f"Rate: {len(batch_ids) / batch_elapsed:.2f} IDs/sec")
        print(f"Batch time: {batch_elapsed:.1f} sec")
        print(f"Total elapsed: {total_elapsed / 60:.1f} min")
        print(f"Last processed ID: {batch_ids[-1]:,}")
        print("-" * 40)


## 14. Final validation


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT media_type, COUNT(*) AS total_records,
               MIN(tmdb_id) AS min_id, MAX(tmdb_id) AS max_id
        FROM bronze.tmdb_metadata
        GROUP BY media_type
        ORDER BY media_type;
    """))

    for row in result:
        print(row)


In [ ]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT source, media_type, export_date,
               last_processed_id, records_processed, last_processed_at
        FROM ops.ingestion_checkpoints
        WHERE source = 'tmdb'
          AND export_date = :export_date
        ORDER BY media_type;
    """), {"export_date": EXPORT_DATE_DB})

    for row in result:
        print(row)


## Performance tuning

Initial settings:
- `MAX_WORKERS = 20`
- `REQUESTS_PER_SECOND = 25`

Increase `REQUESTS_PER_SECOND` gradually only if the benchmark and full batches remain stable.

The goal is not maximum raw request rate. The goal is the highest **stable successful-record throughput** without excessive 429s, timeouts, connection resets, or checkpoint problems.
